# **THỬ NGHIỆM YOLOv9 TRÊN TẬP DỮ LIỆU VISDRONE**

## **TẢI DATASET VISDRONE**
1. Visdrone là bộ dữ liệu dùng để nhận dạng đối tượng từ góc nhìn trên không, được xây dựng bởi Lab of Machine Learning and Data Mining, Tianjin University, China.
2. Mục tiêu: hỗ trợ nghiên cứu về Object Detection, Object Tracking, và Video Analysis trong bối cảnh hình ảnh được quay bằng drone (UAV).
3. Dữ liệu sẽ dùng trong dự án này là: VisDrone2019-DET
* Số lớp: 10 lớp.
* Ảnh train: 6471.
* Ảnh val: 548.
* Ảnh test: 1610 (optional).
* Định dạng nhãn: Yolo txt (lớp + boxx normalized).
* Mục tiêu: Train YOLOv8 để nhận dạng các phương tiện & người trong ảnh chụp từ drone.

### **Bước 1: Tạo file yaml mô tả dữ liệu - dùng để train yolo**

In [1]:
# File yaml mô tả thư mục chứa dữ liệu và các lớp của nó, được lấy từ docs của ultralytics.
import os
from pathlib import Path
import yaml

# Định nghĩa YAML content
yaml_content = """
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# VisDrone2019-DET dataset https://github.com/VisDrone/VisDrone-Dataset by Tianjin University
# Documentation: https://docs.ultralytics.com/datasets/detect/visdrone/
# Example usage: yolo train data=VisDrone.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── VisDrone ← downloads here (2.3 GB)

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: VisDrone # dataset root dir
train: images/train # train images (relative to 'path') 6471 images
val: images/val # val images (relative to 'path') 548 images
test: images/test # test-dev images (optional) 1610 images

# Classes
names:
  0: pedestrian
  1: people
  2: bicycle
  3: car
  4: van
  5: truck
  6: tricycle
  7: awning-tricycle
  8: bus
  9: motor
"""

# Lưu yaml_content vào 1 file
yaml_file_path = 'VisDrone/VisDrone.yaml'
os.makedirs(os.path.dirname(yaml_file_path), exist_ok=True)
with open(yaml_file_path, "w") as f:
  f.write(yaml_content)

### **Bước 2: Tải Dataset và đổi định dạng phù hợp với YOLO**

In [2]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 33.5 MB/s eta 0:00:00


In [3]:
import os
from pathlib import Path
import shutil
from ultralytics.utils.downloads import download
from ultralytics.utils import ASSETS_URL, TQDM

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
# Hàm chuyển đổi định dạng từ visdrone to yolo
def visdrone2yolo(dir, split, source_name=None):
  """Chuyển đổi VisDrone annotations thành định dạng YOLO với cấu trúc images/{split} and labels/{split}."""
  from PIL import Image

  source_dir = dir / (source_name or f"VisDrone2019-DET-{split}")
  images_dir = dir / "images" / split
  labels_dir = dir / "labels" / split
  labels_dir.mkdir(parents=True, exist_ok=True)

  # Chuyển ảnh vào cấu trúc thư mục mới
  if (source_images_dir := source_dir / "images").exists():
    images_dir.mkdir(parents=True, exist_ok=True)
    for img in source_images_dir.glob("*.jpg"):
      img.rename(images_dir / img.name)

  for f in TQDM((source_dir / "annotations").glob("*.txt"), desc=f"Convertiing {split}"):
    img_size = Image.open(images_dir / f.with_suffix(".jpg").name).size
    dw, dh = 1.0 / img_size[0], 1.0 / img_size[1]
    lines = []

    with open(f, encoding="utf-8") as file:
      for row in [x.split(",") for x in file.read().strip().splitlines()]:
        if row[4] == 0:
          continue
        x, y, w, h = map(int, row[:4])
        cls = int(row[5]) - 1 # Index của nhãn (-1 vì nhãn trong yolo bắt đầu từ 0)
        # Chuyển đổi sang YOLO
        x_center, y_center = (x + w / 2) *dw, (y + h / 2) * dh
        w_norm, h_norm = w * dw, h * dh
        lines.append(f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

    (labels_dir / f.name).write_text("".join(lines), encoding="utf-8")

In [5]:
# Tải các file zip và convert
dir = Path("VisDrone")
urls = [
      f"{ASSETS_URL}/VisDrone2019-DET-train.zip",
      f"{ASSETS_URL}/VisDrone2019-DET-val.zip",
      f"{ASSETS_URL}/VisDrone2019-DET-test-dev.zip",]
download(urls, dir=dir, threads=4)

splits = {"VisDrone2019-DET-train": "train", "VisDrone2019-DET-val": "val", "VisDrone2019-DET-test-dev": "test"}
for folder, split in splits.items():
  visdrone2yolo(dir, split, folder)
  shutil.rmtree(dir / folder)

Unzipping VisDrone/VisDrone2019-DET-val.zip to /content/VisDrone/VisDrone2019-DET-val...: 100% ━━━━━━━━━━━━ 1099/1099 1.5Kfiles/s 0.7s
Unzipping VisDrone/VisDrone2019-DET-test-dev.zip to /content/VisDrone/VisDrone2019-DET-test-dev...: 100% ━━━━━━━━━━━━ 3223/3223 998.2files/s 3.2s
Unzipping VisDrone/VisDrone2019-DET-train.zip to /content/VisDrone/VisDrone2019-DET-train...: 100% ━━━━━━━━━━━━ 12945/12945 1.1Kfiles/s 11.7s
Convertiing train: ━━━━━━━━━━━━ 6471 2.4Kit/s 3.5s
Convertiing val: ━━━━━━━━━━━━ 548 1.4Kit/s 0.3s
Convertiing test: ━━━━━━━━━━━━ 1610 2.3Kit/s 0.6s


### **Bước 3: Train YOLO**

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")

In [ ]:
results = model.train(
    data="/content/VisDrone/VisDrone.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    name="yolov8_visdrone",
    device=0
)

Ultralytics 8.3.213 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/VisDrone/VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_visdrone, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=T

### **Bước 4: Lưu model lên gg drive**

In [ ]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

source_dir = '/content/runs/detect/yolov8_visdrone'
destination_dir = '/content/drive/MyDrive/yolov8_visdrone_results'


os.makedirs(destination_dir, exist_ok=True)
shutil.copytree(source_dir, destination_dir, dirs_exist_ok=True)
print(f"Copied '{source_dir}' to '{destination_dir}'")

### **Bước 5: Đóng gói mô hình trên PySpark**

In [59]:
# Mount Google Drive để truy cập dữ liệu và mô hình YOLO
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [60]:
!pip install pyspark # Cài đặt thư viện pyspark

In [78]:
# Khai báo các thư viện cần thiết
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType, ArrayType, FloatType, StructType, StructField, IntegerType
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.functions import predict_batch_udf
import cv2

In [62]:
# Khởi tạo SprarkSession
spark = SparkSession.builder \
    .appName("YOLOv8VisDrone") \
    .getOrCreate()

print("SparkSession created successfully")

SparkSession created successfully


In [63]:
# Load model từ Google Drive
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/yolov8_visdrone_results/weights/best.pt")
print("YOLOv8 model loaded successfully.")

YOLOv8 model loaded successfully.


In [93]:
# Tạo DataFrame chứa đường dẫn ảnh test cho YOLO
image_dir = "/content/VisDrone/images/test/"
image_paths = [os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith('.jpg')]

df = spark.createDataFrame([(p,) for p in image_paths], ["image_path"])
df.show(5, truncate=False)

+---------------------------------------------------------+
|image_path                                               |
+---------------------------------------------------------+
|/content/VisDrone/images/test/9999952_00000_d_0000060.jpg|
|/content/VisDrone/images/test/0000073_05010_d_0000006.jpg|
|/content/VisDrone/images/test/9999952_00000_d_0000227.jpg|
|/content/VisDrone/images/test/9999952_00000_d_0000088.jpg|
|/content/VisDrone/images/test/9999938_00000_d_0000209.jpg|
+---------------------------------------------------------+
only showing top 5 rows



In [92]:
# Định nghĩa schema cho kết quả dự đoán YOLO
schema = ArrayType(StructType([
    StructField("class_id", IntegerType()),
    StructField("confidence", FloatType()),
    StructField("x1", FloatType()),
    StructField("y1", FloatType()),
    StructField("x2", FloatType()),
    StructField("y2", FloatType())
]))

In [65]:
# Định nghĩa hàm UDF dự đoán đối tượng bằng YOLOv8
@pandas_udf(schema)
def yolo_predict(paths: pd.Series) -> pd.Series:
    from ultralytics import YOLO
    results_list = []
    for path in paths:
        pred = model(path)
        boxes = []
        for box in pred[0].boxes:
            boxes.append({
                "class_id": int(box.cls[0]),
                "confidence": float(box.conf[0]),
                "x1": float(box.xyxy[0][0]),
                "y1": float(box.xyxy[0][1]),
                "x2": float(box.xyxy[0][2]),
                "y2": float(box.xyxy[0][3]),
            })
        results_list.append(boxes)
    return pd.Series(results_list)

In [90]:
# Định nghĩa hàm UDF dự đoán theo từng patch ảnh bằng YOLOv8
def predict_patch(paths: pd.Series) -> pd.Series:
    results_list = []

    for path in paths:
        img = cv2.imread(path)
        if img is None:
            results_list.append([])
            continue

        h, w, _ = img.shape
        patch_size = 640
        step = 640

        boxes = []
        for y in range(0, h, step):
            for x in range(0, w, step):
                patch = img[y:y+patch_size, x:x+patch_size]
                if patch.size == 0:
                    continue
                preds = model.predict(patch, verbose=False)
                for box in preds[0].boxes:
                    boxes.append({
                        "class_id": int(box.cls[0]),
                        "confidence": float(box.conf[0]),
                        "x1": float(box.xyxy[0][0]) + x,
                        "y1": float(box.xyxy[0][1]) + y,
                        "x2": float(box.xyxy[0][2]) + x,
                        "y2": float(box.xyxy[0][3]) + y,
                    })
        results_list.append(boxes)

    return pd.Series(results_list)

In [91]:
predict_patch_udf = predict_batch_udf(predict_patch, return_type=schema, batch_size=16)

In [88]:
df = df.withColumn("yolo_detections", yolo_predict(F.col("image_path")))
df.show(truncate=False)

+---------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [76]:
# Đánh giá hiệu suất mô hình YOLOv8 trên tập dữ liệu VisDrone
results = model.val(data="/content/VisDrone/VisDrone.yaml", batch=16)
metrics_dict = results.results_dict
print("mAP@0.5:", metrics_dict['metrics/mAP50(B)'])
print("mAP@0.5:0.95:", metrics_dict['metrics/mAP50-95(B)'])
print("Precision:", metrics_dict['metrics/precision(B)'])
print("Recall:", metrics_dict['metrics/recall(B)'])

Ultralytics 8.3.213 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 900.9±792.7 MB/s, size: 174.3 KB)
val: Scanning /content/VisDrone/labels/val.cache... 548 images, 0 backgrounds, 351 corrupt: 100% ━━━━━━━━━━━━ 548/548 1.3Mit/s 0.0s
val: /content/VisDrone/images/val/0000001_02999_d_0000005.jpg: ignoring corrupt image/label: negative class labels or coordinate [         -1]
val: /content/VisDrone/images/val/0000001_03499_d_0000006.jpg: ignoring corrupt image/label: negative class labels or coordinate [         -1]
val: /content/VisDrone/images/val/0000001_03999_d_0000007.jpg: ignoring corrupt image/label: negative class labels or coordinate [         -1          -1]
val: /content/VisDrone/images/val/0000001_04527_d_0000008.jpg: ignoring corrupt image/label: negative class labels or coordinate [         -1          -1          -1          -1          -1          -1          -1          -1          -1]
val: /content/VisDron